# (Des)carga de base de datos

## Instalación y carga de la librería del repositorio UCI ML

Instalar la librería del repositorio utilizando pip.

In [ ]:
pip install ucimlrepo

Verificar que la librería del repositorio se encuentra correctamente instalada.

In [ ]:
pip list | grep uci

Cargar la librería en la sesión actual de python.

In [1]:
from ucimlrepo import fetch_ucirepo

## Carga de la base de datos de cirrosis

Según la documentación del repositorio, nuestra base de datos de interés (cirrosis) se encuentra indexada con el id 878.

In [2]:
cirrhosis_patient_survival_prediction = fetch_ucirepo(id=878)

Ver cómo viene la información

In [ ]:
cirrhosis_patient_survival_prediction

Estructurar la información en un data frame

In [ ]:
X = cirrhosis_patient_survival_prediction.data.features
y = cirrhosis_patient_survival_prediction.data.targets
cirrosis = pd.concat([X,y], axis=1)

Ver las primeras observaciones

In [ ]:
cirrosis.head(10)

## En caso que fracasemos con la librería uci

Cargar librerías auxiliares

In [3]:
# Importar clase Path para utilizar rutas en directorios
from pathlib import Path
# Importar pandas para cargar base como dataframe
import pandas as pd

Ruta de trabajo actual para movernos entre directorios y poder leer la base correctamente.

In [4]:
BASE_DIR = Path().resolve().parent

Ver cual es el directorio al que apunta BASE_DIR

In [ ]:
BASE_DIR

Ruta de del directorio donde se encuentra el archivo de cirrosis 

In [5]:
DATA_DIR = BASE_DIR / 'data' / 'cirrosis'
DATA_DIR

PosixPath('/Users/davidmoreno/Desktop/Diplomado 2025/DMEMCDM1/data/cirrosis')

Leer el archivo csv que se encuentra en el directorio de cirrosis utilizando pandas para estructurarlo en un data frame.

In [ ]:
cirrosis2 = pd.read_csv(DATA_DIR / 'cirrhosis.csv')

Ver las primeros renglones del data frame.

In [ ]:
cirrosis2.head(10)

## Sanity check

Verificar que tienen la misma dimensión

In [ ]:
cirrosis.shape

In [ ]:
cirrosis2.shape

Observamos que en cirrosis2 existen un par de columnas que no se encontraban en cirrosis:

In [ ]:
set(cirrosis2.columns).symmetric_difference(set(cirrosis.columns))

La columna 'ID' sirve de identificador por lo cual no representa ninguna diferencia significativa que pudiera afectar el análisis de nuestros datos. ¿Pero que pasa con 'N_Days'?

**Revisando la documentación** encontramos que dicha variable representaba el tiempo transcurrido desde que se registraban al estudio a que pasaba uno de los siguientes escenarios:
1. Muerte (D)
2. Trasplante (CL)
3. Fin del estudio (C)

Cómo nosotros no estamos haciendo el análisis de supervivencia y tomamos los datos con fines meramente de estudio, vamos a considerar N_Days también como variable explicativa.

## "Arreglo" de las bases

Primero voy a buscar como agregar la columna N_Days a mi base cirrosis. Explorando el objeto cirrhosis_patient_survival_prediction vemos que en data había un original que es idéntica a la tabla generada con cirrosis2.

In [6]:
cirrosis = cirrhosis_patient_survival_prediction.data.original
cirrosis.head(10)

,ID,N_Days,Status,Drug,Age,Sex,Ascites,Hepatomegaly,Spiders,Edema,Bilirubin,Cholesterol,Albumin,Copper,Alk_Phos,SGOT,Tryglicerides,Platelets,Prothrombin,Stage
0,1,400,D,D-penicillamine,21464,F,Y,Y,Y,Y,14.5,261,2.60,156,1718.0,137.95,172,190,12.2,4.0
1,2,4500,C,D-penicillamine,20617,F,N,Y,Y,N,1.1,302,4.14,54,7394.8,113.52,88,221,10.6,3.0
2,3,1012,D,D-penicillamine,25594,M,N,N,N,S,1.4,176,3.48,210,516.0,96.10,55,151,12.0,4.0
3,4,1925,D,D-penicillamine,19994,F,N,Y,Y,S,1.8,244,2.54,64,6121.8,60.63,92,183,10.3,4.0
4,5,1504,CL,Placebo,13918,F,N,Y,Y,N,3.4,279,3.53,143,671.0,113.15,72,136,10.9,3.0
5,6,2503,D,Placebo,24201,F,N,Y,N,N,0.8,248,3.98,50,944.0,93.00,63,NaNN,11.0,3.0
6,7,1832,C,Placebo,20284,F,N,Y,N,N,1.0,322,4.09,52,824.0,60.45,213,204,9.7,3.0
7,8,2466,D,Placebo,19379,F,N,N,N,N,0.3,280,4.00,52,4651.2,28.38,189,373,11.0,3.0
8,9,2400,D,D-penicillamine,15526,F,N,N,Y,N,3.2,562,3.08,79,2276.0,144.15,88,251,11.0,2.0
9,10,51,D,Placebo,25772,F,Y,N,Y,Y,12.6,200,2.74,140,918.0,147.25,143,302,11.5,4.0


In [ ]:
all(cirrosis == cirrosis2)

De esta manera, podemos quedarnos solo con cirrosis, "descartando" la columna 'ID', y eliminar el objeto cirrosis2.

OJO: Dependiendo de la base que pudieron descargar, a esa le "descartan" la columna 'ID' y la otra no existía entonces no la borran.

In [7]:
cirrosis.set_index('ID', inplace=True)
cirrosis.head(2)

,N_Days,Status,Drug,Age,Sex,Ascites,Hepatomegaly,Spiders,Edema,Bilirubin,Cholesterol,Albumin,Copper,Alk_Phos,SGOT,Tryglicerides,Platelets,Prothrombin,Stage
ID,,,,,,,,,,,,,,,,,,,
1,400,D,D-penicillamine,21464,F,Y,Y,Y,Y,14.5,261,2.60,156,1718.0,137.95,172,190,12.2,4.0
2,4500,C,D-penicillamine,20617,F,N,Y,Y,N,1.1,302,4.14,54,7394.8,113.52,88,221,10.6,3.0


In [ ]:
del cirrosis2

Reordenaremos las columnas para tener la variable de interés (Status) al final.

In [8]:
cirrosis = cirrosis[[x for x in cirrosis.columns if x != 'Status'] + ['Status']]
cirrosis.head()

,N_Days,Drug,Age,Sex,Ascites,Hepatomegaly,Spiders,Edema,Bilirubin,Cholesterol,Albumin,Copper,Alk_Phos,SGOT,Tryglicerides,Platelets,Prothrombin,Stage,Status
ID,,,,,,,,,,,,,,,,,,,
1,400,D-penicillamine,21464,F,Y,Y,Y,Y,14.5,261,2.60,156,1718.0,137.95,172,190,12.2,4.0,D
2,4500,D-penicillamine,20617,F,N,Y,Y,N,1.1,302,4.14,54,7394.8,113.52,88,221,10.6,3.0,C
3,1012,D-penicillamine,25594,M,N,N,N,S,1.4,176,3.48,210,516.0,96.10,55,151,12.0,4.0,D
4,1925,D-penicillamine,19994,F,N,Y,Y,S,1.8,244,2.54,64,6121.8,60.63,92,183,10.3,4.0,D
5,1504,Placebo,13918,F,N,Y,Y,N,3.4,279,3.53,143,671.0,113.15,72,136,10.9,3.0,CL


# Filtrado, ordenamiento, mezclas y agregación

## Filtrado de filas de acuerdo a condiciones lógicas

En ocasiones nos interesan ciertas observaciones por lo que necesitamos sera capaces de extraerlas. Veamos las distintas etapas que existen para luego filtrar las observaciones correspondientes con la etapa inicial de la enfermedad.

In [9]:
cirrosis.Stage.value_counts()

Stage
3.0    155
4.0    144
2.0     92
1.0     21
Name: count, dtype: int64

In [10]:
cirrosis_etapa_1 = cirrosis[cirrosis['Stage'] < 2]
cirrosis_etapa_1.head(5)

,N_Days,Drug,Age,Sex,Ascites,Hepatomegaly,Spiders,Edema,Bilirubin,Cholesterol,Albumin,Copper,Alk_Phos,SGOT,Tryglicerides,Platelets,Prothrombin,Stage,Status
ID,,,,,,,,,,,,,,,,,,,
52,2386,D-penicillamine,18460,M,N,N,N,N,6.0,614,3.70,158,5084.4,206.40,93,362,10.6,1.0,D
58,4459,D-penicillamine,16279,M,N,N,N,N,0.7,242,4.08,73,5890.0,56.76,118,NaNN,10.6,1.0,C
61,4256,Placebo,16034,M,N,N,N,N,0.6,216,3.94,28,601.0,60.45,188,211,13.0,1.0,C
65,3992,D-penicillamine,14684,F,N,N,N,N,1.2,256,3.60,74,724.0,141.05,108,430,10.0,1.0,C
73,4190,Placebo,14060,F,N,N,N,N,0.7,132,3.60,17,423.0,49.60,56,265,11.0,1.0,C


In [12]:
cirrosis_etapa_1.Stage.value_counts()

Stage
1.0    21
Name: count, dtype: int64

OJO: Cuidado con el tipo de dato que tienen porque en ocasiones al no tomarlo en consideración las operaciones y los filtros pueden no comportarse de manera adecuada.

In [17]:
cirrosis[cirrosis['Stage'] == '1.0']

,N_Days,Drug,Age,Sex,Ascites,Hepatomegaly,Spiders,Edema,Bilirubin,Cholesterol,Albumin,Copper,Alk_Phos,SGOT,Tryglicerides,Platelets,Prothrombin,Stage,Status
ID,,,,,,,,,,,,,,,,,,,


En ocasiones se pueden necesitar filtros que dependan de más de una variables; por ejemplo, menores de edad en etapa 1 de género F. Hay que tener cuidado pues la edad esta en días.

In [18]:
stage_mask = (cirrosis['Stage'] < 2)
stage_mask

ID
1      False
2      False
3      False
4      False
5      False
       ...  
414    False
415    False
416    False
417    False
418    False
Name: Stage, Length: 418, dtype: bool

Para verificar que funciona bien la máscara, pueden sumar los boleanos y deberían ser 21

In [19]:
stage_mask.sum()

np.int64(21)

In [21]:
age_mask = (cirrosis['Age'] / 365 < 18)
sex_mask = (cirrosis['Sex'] == 'F')

Filtro las observaciones que cumplan con todos los requisitos.

In [22]:
cirrosis_F_menores_edad = cirrosis[stage_mask & age_mask & sex_mask]

Equivalentemente podrían escribirlo como sigue

In [ ]:
cirrosis[(cirrosis['Stage'] < 2) & (cirrosis['Age'] / 365 < 18) & (cirrosis['Sex'] == 'F')]

,N_Days,Drug,Age,Sex,Ascites,Hepatomegaly,Spiders,Edema,Bilirubin,Cholesterol,Albumin,Copper,Alk_Phos,SGOT,Tryglicerides,Platelets,Prothrombin,Stage,Status
ID,,,,,,,,,,,,,,,,,,,


In [27]:
cirrosis_F_menores_edad

,N_Days,Drug,Age,Sex,Ascites,Hepatomegaly,Spiders,Edema,Bilirubin,Cholesterol,Albumin,Copper,Alk_Phos,SGOT,Tryglicerides,Platelets,Prothrombin,Stage,Status
ID,,,,,,,,,,,,,,,,,,,


Observamos que en el estudio no hay menores de edad

In [28]:
(cirrosis['Age'] / 365 < 18).sum()

np.int64(0)

Nos enteramos que no es legal hacer estudios clínicos experimentales en niños (sin consentimiento de sus tutores legales y en casos no extremos) por lo cual nuestra base no tiene información de menores de edad.

Bueno... vamos a relajar un poco el supuesto y digamos que queremos menores de edad o mayores de 65 años.

In [29]:
age_mask_2 = age_mask | (cirrosis['Age'] / 365 > 65)
cirrosis_F_no_laboral = cirrosis[stage_mask & sex_mask & age_mask_2]

Equivalentemente...

In [30]:
cirrosis[(cirrosis['Stage'] < 2) & ((cirrosis['Age'] / 365 < 18) | (cirrosis['Age'] / 365 > 65)) & (cirrosis['Sex'] == 'F')]

,N_Days,Drug,Age,Sex,Ascites,Hepatomegaly,Spiders,Edema,Bilirubin,Cholesterol,Albumin,Copper,Alk_Phos,SGOT,Tryglicerides,Platelets,Prothrombin,Stage,Status
ID,,,,,,,,,,,,,,,,,,,


In [32]:
cirrosis_F_no_laboral

,N_Days,Drug,Age,Sex,Ascites,Hepatomegaly,Spiders,Edema,Bilirubin,Cholesterol,Albumin,Copper,Alk_Phos,SGOT,Tryglicerides,Platelets,Prothrombin,Stage,Status
ID,,,,,,,,,,,,,,,,,,,


Haciendo un análisis de la información vemos que el rango de edad en pacientes género 'F' que se encuentran en etapa 1 esta entre 28 y 62 años. Por lo cual hace sentido que no recuperemos información.

In [36]:
(cirrosis[stage_mask].Age /365).describe()

count    21.000000
mean     46.873190
std       9.552225
min      28.904110
25%      38.520548
50%      46.380822
75%      53.035616
max      62.564384
Name: Age, dtype: float64

In [37]:
(cirrosis[sex_mask].Age /365).describe()

count    374.000000
mean      50.191297
std       10.247664
min       26.295890
25%       42.407534
50%       50.227397
75%       57.038356
max       76.761644
Name: Age, dtype: float64

In [38]:
(cirrosis[age_mask_2].Age /365).describe()

count    40.000000
mean     69.141644
std       3.251015
min      65.043836
25%      67.046575
50%      68.278082
75%      70.242466
max      78.493151
Name: Age, dtype: float64

## Selección de variables (columnas)

In [39]:
cirrosis.head(2)

,N_Days,Drug,Age,Sex,Ascites,Hepatomegaly,Spiders,Edema,Bilirubin,Cholesterol,Albumin,Copper,Alk_Phos,SGOT,Tryglicerides,Platelets,Prothrombin,Stage,Status
ID,,,,,,,,,,,,,,,,,,,
1,400,D-penicillamine,21464,F,Y,Y,Y,Y,14.5,261,2.60,156,1718.0,137.95,172,190,12.2,4.0,D
2,4500,D-penicillamine,20617,F,N,Y,Y,N,1.1,302,4.14,54,7394.8,113.52,88,221,10.6,3.0,C


In [40]:
cirrosis.columns

Index(['N_Days', 'Drug', 'Age', 'Sex', 'Ascites', 'Hepatomegaly', 'Spiders',
       'Edema', 'Bilirubin', 'Cholesterol', 'Albumin', 'Copper', 'Alk_Phos',
       'SGOT', 'Tryglicerides', 'Platelets', 'Prothrombin', 'Stage', 'Status'],
      dtype='object')

Guardar nombres de distintas variables en listas dependiendo de su tipo o significado.

In [41]:
signos_clinicos = ['Ascites', 'Hepatomegaly', 'Spiders', 'Edema']
nivels_sangre = ['Bilirubin', 'Cholesterol', 'Albumin', 'Copper', 'Alk_Phos', 'SGOT', 'Tryglicerides', 'Platelets', 'Prothrombin']
variables_categoricas_nominales = ['Drug', 'Sex', 'Status']
variables_categoricas_ordinales = ['Stage']
variables_continuas = ['N_Days', 'Age']

Sanity check que no elimine variables...

In [42]:
len(signos_clinicos) + len(nivels_sangre) + len(variables_categoricas_nominales) + len(variables_categoricas_ordinales) + len(variables_continuas) == len(cirrosis.columns)

True

Qué pasa si de momento sólo nos interesan los signos clínicos.

In [45]:
cirrosis_signos_clínicos = cirrosis[signos_clinicos]
cirrosis_signos_clínicos.head()

,Ascites,Hepatomegaly,Spiders,Edema
ID,,,,
1,Y,Y,Y,Y
2,N,Y,Y,N
3,N,N,N,S
4,N,Y,Y,S
5,N,Y,Y,N


Equivalentemente se podría hacer de la siguiente manera:

In [46]:
cirrosis[['Ascites', 'Hepatomegaly', 'Spiders', 'Edema']]

,Ascites,Hepatomegaly,Spiders,Edema
ID,,,,
1,Y,Y,Y,Y
2,N,Y,Y,N
3,N,N,N,S
4,N,Y,Y,S
5,N,Y,Y,N
...,...,...,...,...
414,NaN,NaN,NaN,N
415,NaN,NaN,NaN,N
416,NaN,NaN,NaN,N


De igual manera, podríamos querer las variables de signo clínico y de niveles en sangre.

In [48]:
cirrosis_variables_médicas = cirrosis[signos_clinicos + nivels_sangre]
cirrosis_variables_médicas.head()

,Ascites,Hepatomegaly,Spiders,Edema,Bilirubin,Cholesterol,Albumin,Copper,Alk_Phos,SGOT,Tryglicerides,Platelets,Prothrombin
ID,,,,,,,,,,,,,
1,Y,Y,Y,Y,14.5,261,2.60,156,1718.0,137.95,172,190,12.2
2,N,Y,Y,N,1.1,302,4.14,54,7394.8,113.52,88,221,10.6
3,N,N,N,S,1.4,176,3.48,210,516.0,96.10,55,151,12.0
4,N,Y,Y,S,1.8,244,2.54,64,6121.8,60.63,92,183,10.3
5,N,Y,Y,N,3.4,279,3.53,143,671.0,113.15,72,136,10.9


In [49]:
signos_clinicos + nivels_sangre

['Ascites',
 'Hepatomegaly',
 'Spiders',
 'Edema',
 'Bilirubin',
 'Cholesterol',
 'Albumin',
 'Copper',
 'Alk_Phos',
 'SGOT',
 'Tryglicerides',
 'Platelets',
 'Prothrombin']

Equivalentemente...

In [50]:
cirrosis[['Ascites', 'Hepatomegaly', 'Spiders', 'Edema', 'Bilirubin', 'Cholesterol', 'Albumin', 'Copper', 'Alk_Phos', 'SGOT', 'Tryglicerides', 'Platelets', 'Prothrombin']]

,Ascites,Hepatomegaly,Spiders,Edema,Bilirubin,Cholesterol,Albumin,Copper,Alk_Phos,SGOT,Tryglicerides,Platelets,Prothrombin
ID,,,,,,,,,,,,,
1,Y,Y,Y,Y,14.5,261,2.60,156,1718.0,137.95,172,190,12.2
2,N,Y,Y,N,1.1,302,4.14,54,7394.8,113.52,88,221,10.6
3,N,N,N,S,1.4,176,3.48,210,516.0,96.10,55,151,12.0
4,N,Y,Y,S,1.8,244,2.54,64,6121.8,60.63,92,183,10.3
5,N,Y,Y,N,3.4,279,3.53,143,671.0,113.15,72,136,10.9
...,...,...,...,...,...,...,...,...,...,...,...,...,...
414,NaN,NaN,NaN,N,1.2,NaN,2.96,NaN,NaN,NaN,NaN,174,10.9
415,NaN,NaN,NaN,N,0.9,NaN,3.83,NaN,NaN,NaN,NaN,180,11.2
416,NaN,NaN,NaN,N,1.6,NaN,3.42,NaN,NaN,NaN,NaN,143,9.9


Hack:

In [51]:
cirrosis[[
    'Ascites', 'Hepatomegaly', 'Spiders', 'Edema', 'Bilirubin',
    'Cholesterol', 'Albumin', 'Copper', 'Alk_Phos', 'SGOT', 'Tryglicerides', 'Platelets', 'Prothrombin'
]]

,Ascites,Hepatomegaly,Spiders,Edema,Bilirubin,Cholesterol,Albumin,Copper,Alk_Phos,SGOT,Tryglicerides,Platelets,Prothrombin
ID,,,,,,,,,,,,,
1,Y,Y,Y,Y,14.5,261,2.60,156,1718.0,137.95,172,190,12.2
2,N,Y,Y,N,1.1,302,4.14,54,7394.8,113.52,88,221,10.6
3,N,N,N,S,1.4,176,3.48,210,516.0,96.10,55,151,12.0
4,N,Y,Y,S,1.8,244,2.54,64,6121.8,60.63,92,183,10.3
5,N,Y,Y,N,3.4,279,3.53,143,671.0,113.15,72,136,10.9
...,...,...,...,...,...,...,...,...,...,...,...,...,...
414,NaN,NaN,NaN,N,1.2,NaN,2.96,NaN,NaN,NaN,NaN,174,10.9
415,NaN,NaN,NaN,N,0.9,NaN,3.83,NaN,NaN,NaN,NaN,180,11.2
416,NaN,NaN,NaN,N,1.6,NaN,3.42,NaN,NaN,NaN,NaN,143,9.9


## Ordenamiento

Tal vez quisieramos ordenar la información de acuerdo a la edad de los pacientes...

In [55]:
cirrosis_orden_edad_decreciente = cirrosis.sort_values(by='Age', ascending=False)
cirrosis_orden_edad_decreciente

,N_Days,Drug,Age,Sex,Ascites,Hepatomegaly,Spiders,Edema,Bilirubin,Cholesterol,Albumin,Copper,Alk_Phos,SGOT,Tryglicerides,Platelets,Prothrombin,Stage,Status
ID,,,,,,,,,,,,,,,,,,,
253,1765,D-penicillamine,28650,M,Y,Y,Y,N,7.1,243,3.03,380,983.0,158.10,154,97,11.2,4.0,C
92,388,D-penicillamine,28018,F,Y,N,N,Y,1.4,206,3.13,36,1626.0,86.80,70,145,12.2,4.0,D
147,2995,D-penicillamine,27398,F,N,N,N,S,1.2,288,3.37,32,791.0,57.35,114,213,10.7,2.0,C
316,2071,NaN,27394,F,NaN,NaN,NaN,S,0.7,NaN,3.96,NaN,NaN,NaN,NaN,NaN,11.3,4.0,D
260,1656,Placebo,27220,M,N,Y,N,N,5.6,232,3.59,188,1120.0,98.00,128,248,10.9,4.0,C
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
307,1149,Placebo,11167,F,N,N,N,N,0.8,273,3.56,52,1282.0,130.00,59,344,10.5,2.0,C
173,2657,D-penicillamine,11058,F,N,Y,Y,N,3.0,236,3.42,76,1403.0,89.90,86,493,9.8,2.0,C
195,2330,D-penicillamine,10795,F,N,Y,N,N,3.7,347,3.90,76,2544.0,221.65,90,129,11.5,4.0,C


In [56]:
cirrosis_orden_edad_creciente = cirrosis.sort_values(by='Age', ascending=True)
cirrosis_orden_edad_creciente

,N_Days,Drug,Age,Sex,Ascites,Hepatomegaly,Spiders,Edema,Bilirubin,Cholesterol,Albumin,Copper,Alk_Phos,SGOT,Tryglicerides,Platelets,Prothrombin,Stage,Status
ID,,,,,,,,,,,,,,,,,,,
270,1568,D-penicillamine,9598,F,N,Y,Y,N,1.0,448,3.74,102,1128.0,71.00,117,228,10.2,3.0,C
98,3823,D-penicillamine,10550,F,N,N,N,N,1.0,239,3.77,77,1877.0,97.65,101,312,10.2,1.0,C
195,2330,D-penicillamine,10795,F,N,Y,N,N,3.7,347,3.90,76,2544.0,221.65,90,129,11.5,4.0,C
173,2657,D-penicillamine,11058,F,N,Y,Y,N,3.0,236,3.42,76,1403.0,89.90,86,493,9.8,2.0,C
307,1149,Placebo,11167,F,N,N,N,N,0.8,273,3.56,52,1282.0,130.00,59,344,10.5,2.0,C
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
260,1656,Placebo,27220,M,N,Y,N,N,5.6,232,3.59,188,1120.0,98.00,128,248,10.9,4.0,C
316,2071,NaN,27394,F,NaN,NaN,NaN,S,0.7,NaN,3.96,NaN,NaN,NaN,NaN,NaN,11.3,4.0,D
147,2995,D-penicillamine,27398,F,N,N,N,S,1.2,288,3.37,32,791.0,57.35,114,213,10.7,2.0,C


Y si me interesa ordenar de acuerdo a varias variables?

Por ejemplo, nos intersa ordenar por etapa y número de días desde diagnóstico a el suceso..

In [57]:
cirrosis_orden_etapa_creciente_dias_creciente = cirrosis.sort_values(by=['Stage', 'N_Days'])
cirrosis_orden_etapa_creciente_dias_creciente

,N_Days,Drug,Age,Sex,Ascites,Hepatomegaly,Spiders,Edema,Bilirubin,Cholesterol,Albumin,Copper,Alk_Phos,SGOT,Tryglicerides,Platelets,Prothrombin,Stage,Status
ID,,,,,,,,,,,,,,,,,,,
371,489,NaN,18628,F,NaN,NaN,NaN,S,7.3,NaN,3.52,NaN,NaN,NaN,NaN,265,11.1,1.0,D
395,1329,NaN,13149,F,NaN,NaN,NaN,N,1.4,NaN,3.98,NaN,NaN,NaN,NaN,402,11.0,1.0,C
285,1401,D-penicillamine,16929,F,N,N,N,N,0.8,253,3.48,65,688.0,57.0,80,252,10.0,1.0,C
272,1525,D-penicillamine,14025,F,N,N,N,N,0.5,226,2.93,22,674.0,58.0,85,153,9.8,1.0,C
384,1639,NaN,21550,F,NaN,NaN,NaN,N,1.3,NaN,3.40,NaN,NaN,NaN,NaN,243,9.7,1.0,C
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
334,466,NaN,20454,F,NaN,NaN,NaN,N,7.1,NaN,3.51,NaN,NaN,NaN,NaN,721,11.8,NaN,D
322,2011,NaN,23376,F,NaN,NaN,NaN,N,1.1,NaN,3.69,NaN,NaN,NaN,NaN,139,10.5,NaN,D
337,2286,NaN,20454,F,NaN,NaN,NaN,N,1.8,NaN,3.64,NaN,NaN,NaN,NaN,141,10.0,NaN,D


Como verán no vemos todas las filas por lo que hay que modificar los parámetros de pandas

In [58]:
pd.set_option('display.max_rows', None)

In [59]:
cirrosis_orden_etapa_creciente_dias_creciente

,N_Days,Drug,Age,Sex,Ascites,Hepatomegaly,Spiders,Edema,Bilirubin,Cholesterol,Albumin,Copper,Alk_Phos,SGOT,Tryglicerides,Platelets,Prothrombin,Stage,Status
ID,,,,,,,,,,,,,,,,,,,
371,489,NaN,18628,F,NaN,NaN,NaN,S,7.3,NaN,3.52,NaN,NaN,NaN,NaN,265,11.1,1.0,D
395,1329,NaN,13149,F,NaN,NaN,NaN,N,1.4,NaN,3.98,NaN,NaN,NaN,NaN,402,11.0,1.0,C
285,1401,D-penicillamine,16929,F,N,N,N,N,0.8,253,3.48,65,688.0,57.00,80,252,10.0,1.0,C
272,1525,D-penicillamine,14025,F,N,N,N,N,0.5,226,2.93,22,674.0,58.00,85,153,9.8,1.0,C
384,1639,NaN,21550,F,NaN,NaN,NaN,N,1.3,NaN,3.40,NaN,NaN,NaN,NaN,243,9.7,1.0,C
258,1702,D-penicillamine,18806,F,N,N,N,N,1.1,414,3.44,80,1003.0,99.00,55,271,9.6,1.0,C
218,2170,D-penicillamine,12636,F,N,N,N,N,0.5,NaNN,3.89,29,897.0,66.65,NaNN,423,10.1,1.0,C
206,2255,D-penicillamine,22642,F,N,N,N,N,0.6,213,4.07,12,5300.0,57.35,68,240,11.0,1.0,C
52,2386,D-penicillamine,18460,M,N,N,N,N,6.0,614,3.70,158,5084.4,206.40,93,362,10.6,1.0,D


¿Qué pasa si quiero el número de días en orden decreciente y la etapa en orden creciente?

In [60]:
cirrosis_orden_etapa_creciente_dias_decreciente = cirrosis.sort_values(by=['Stage', 'N_Days'], ascending=[True, False])
cirrosis_orden_etapa_creciente_dias_decreciente.head(5)

,N_Days,Drug,Age,Sex,Ascites,Hepatomegaly,Spiders,Edema,Bilirubin,Cholesterol,Albumin,Copper,Alk_Phos,SGOT,Tryglicerides,Platelets,Prothrombin,Stage,Status
ID,,,,,,,,,,,,,,,,,,,
58,4459,D-penicillamine,16279,M,N,N,N,N,0.7,242,4.08,73,5890.0,56.76,118,NaNN,10.6,1.0,C
61,4256,Placebo,16034,M,N,N,N,N,0.6,216,3.94,28,601.0,60.45,188,211,13.0,1.0,C
73,4190,Placebo,14060,F,N,N,N,N,0.7,132,3.60,17,423.0,49.60,56,265,11.0,1.0,C
65,3992,D-penicillamine,14684,F,N,N,N,N,1.2,256,3.60,74,724.0,141.05,108,430,10.0,1.0,C
98,3823,D-penicillamine,10550,F,N,N,N,N,1.0,239,3.77,77,1877.0,97.65,101,312,10.2,1.0,C


¿Afecta el orden en el by? Sí, es la jerarquía dijera José...

In [64]:
cirrosis_orden_dias_decreciente_etapa_creciente = cirrosis.sort_values(by=['N_Days', 'Stage'], ascending=[False, True])
cirrosis_orden_dias_decreciente_etapa_creciente

,N_Days,Drug,Age,Sex,Ascites,Hepatomegaly,Spiders,Edema,Bilirubin,Cholesterol,Albumin,Copper,Alk_Phos,SGOT,Tryglicerides,Platelets,Prothrombin,Stage,Status
ID,,,,,,,,,,,,,,,,,,,
325,4795,NaN,12419,F,NaN,NaN,NaN,N,1.8,NaN,3.24,NaN,NaN,NaN,NaN,NaN,18.0,2.0,C
43,4556,D-penicillamine,17850,F,N,N,N,N,1.1,361,3.64,36,5430.2,67.08,89,203,10.6,2.0,C
32,4523,Placebo,19722,F,N,Y,N,N,1.8,262,3.34,101,7277.0,82.56,158,286,10.6,4.0,C
29,4509,Placebo,23331,F,N,N,N,N,0.7,370,3.78,24,5833.0,73.53,86,390,10.6,2.0,C
2,4500,D-penicillamine,20617,F,N,Y,Y,N,1.1,302,4.14,54,7394.8,113.52,88,221,10.6,3.0,C
40,4467,D-penicillamine,17046,F,N,N,N,N,1.3,NaNN,3.34,105,11046.6,104.49,NaNN,358,11.0,4.0,C
58,4459,D-penicillamine,16279,M,N,N,N,N,0.7,242,4.08,73,5890.0,56.76,118,NaNN,10.6,1.0,C
42,4453,Placebo,12307,F,N,Y,Y,N,2.1,NaNN,3.54,122,8778.0,56.76,NaNN,344,11.0,4.0,C
48,4427,Placebo,17947,M,N,N,N,N,1.9,259,3.70,281,10396.8,188.34,178,214,11.0,3.0,C
